In [ ]:
import pandas as pd

r = pd.read_json("../data/processed/records.jsonl", lines=True)
t = r[r.split == "test"]            # held-out students only, so the numbers are honest

def summ(g):
    a = g.actual_at_risk == 1
    return pd.Series({
        "students": len(g),
        "at_risk_rate": a.mean(),
        "recall": (g.flagged & a).sum() / max(a.sum(), 1),
        "precision": (g.flagged & a).sum() / max(g.flagged.sum(), 1)})

t.groupby(["cutoff_day", "module"]).apply(summ, include_groups=False).round(3)

In [ ]:
res = t.groupby(["cutoff_day", "module"]).apply(summ, include_groups=False).round(3)
res.to_csv("../reports/recall_by_module.csv")

In [ ]:
import pandas as pd

r = pd.read_json("../data/processed/records.jsonl", lines=True)
ti = pd.json_normalize(r["theme_impacts"].tolist()).add_prefix("t_")
r = pd.concat([r.drop(columns="theme_impacts"), ti], axis=1)
r["bg_driven"] = (r.t_background > 0) & (r.t_background > r.t_engagement + r.t_assessments)
test = r[r.split == "test"]                       # held-out students only

# A) flags driven mainly by background, and how often they were right
print(test[test.flagged].groupby("cutoff_day").apply(lambda g: pd.Series({
    "flagged": len(g),
    "background_driven_share": g.bg_driven.mean(),
    "right_if_background_driven": (g.actual_at_risk[g.bg_driven] == 1).mean(),
    "right_if_behaviour_driven": (g.actual_at_risk[~g.bg_driven] == 1).mean()}),
    include_groups=False).round(3))

# B) recall and false-alarm rate by group
info = pd.read_csv("../data/raw/studentInfo.csv")[
    ["code_module", "code_presentation", "id_student", "gender", "disability", "imd_band"]]
m = test.merge(info, left_on=["module", "presentation", "id_student"],
               right_on=["code_module", "code_presentation", "id_student"], how="left")
m["imd_band"] = m["imd_band"].fillna("Unknown")

def rates(g):
    a = g.actual_at_risk == 1
    return pd.Series({"students": len(g), "at_risk_rate": a.mean(),
                      "recall": (g.flagged & a).sum() / max(a.sum(), 1),
                      "false_alarm_rate": (g.flagged & ~a).sum() / max((~a).sum(), 1)})

for col in ["gender", "disability", "imd_band"]:
    print(m.groupby(["cutoff_day", col]).apply(rates, include_groups=False).round(3), "\n")

In [ ]:
import pandas as pd
pd.DataFrame({
    "cutoff":    [28, 56, 84],
    "train_n":   [9430, 9139, 8911],
    "test_n":    [15729, 15092, 14590],
    "auc_2014":  [0.753, 0.806, 0.829],
    "threshold": [0.39, 0.36, 0.35],
    "precision": [0.633, 0.646, 0.644],
    "recall":    [0.713, 0.741, 0.766]}).to_csv("../reports/results_2013_to_2014.csv", index=False)b

In [ ]:
import pandas as pd
rows = []
data = {  # module: students at day 28, at-risk rate at day 28, recall x3, precision x3
    "AAA": (148, 0.284, [0.452, 0.538, 0.541], [0.594, 0.618, 0.625]),
    "BBB": (1275, 0.413, [0.629, 0.650, 0.658], [0.574, 0.620, 0.630]),
    "CCC": (740, 0.508, [0.840, 0.806, 0.800], [0.646, 0.673, 0.695]),
    "DDD": (1053, 0.539, [0.796, 0.773, 0.751], [0.706, 0.746, 0.757]),
    "EEE": (561, 0.369, [0.671, 0.607, 0.626], [0.565, 0.669, 0.699]),
    "FFF": (1253, 0.448, [0.791, 0.784, 0.793], [0.652, 0.687, 0.676]),
    "GGG": (506, 0.366, [0.616, 0.794, 0.640], [0.465, 0.483, 0.582])}
for m, (n, ar, rec, prec) in data.items():
    for i, d in enumerate([28, 56, 84]):
        rows.append(dict(module=m, cutoff_day=d, students_day28=n, at_risk_rate_day28=ar,
                         recall=rec[i], precision=prec[i]))
pd.DataFrame(rows).to_csv("../reports/recall_by_module.csv", index=False)